<a href="https://colab.research.google.com/github/sotetsuk/LectureColab/blob/main/reinforce_cartpole.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 応用機械学習 #3：CartPole で REINFORCE を動かす

講義資料「REINFORCEの実装、価値予測、Actor–Critic」の CartPole の実験を、上から順に実行して確かめるノートブックである。

1. CartPole 環境を動かし、ランダムな方策の動きをアニメーションで見る
2. REINFORCE（ベースラインなし）を実装して学習させる
3. 価値 $V_\phi$ を MC 収益への回帰で学習し、ベースラインに使う
4. critic の学習ターゲットを TD に変える

環境も方策も PyTorch だけで書く。
規模が小さいので GPU は要らず、ランタイムは CPU のままでよい。
メニューの「ランタイム」→「すべてのセルを実行」で、上から順に実行できる。
全体の実行には、Colab の CPU ランタイムで15分ほどかかる（学習が6回ある）。

## 0. 準備

`N_SEEDS` は比較に使う乱数シードの数である。
講義資料の図は5シードで描いたが、Colab の CPU では1シードの学習に2分前後かかるので、2にしてある。
時間に余裕があれば増やす。

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from IPython.display import HTML
from matplotlib import animation
from torch.distributions import Categorical

N_SEEDS = 2    # 比較に使う乱数シードの数
UPDATES = 300  # 更新回数

# 図の色と線種 (講義資料と同じ): ベースラインなしは青の実線, MC は朱の破線, TD は緑の点線
COLOR = {"none": "#0072B2", "mc": "#D55E00", "td": "#009E73"}
LS = {"none": "-", "mc": "--", "td": ":"}
LABEL = {"none": "no baseline", "mc": "value baseline (MC)", "td": "value baseline (TD)"}
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "animation.embed_limit": 100})

## 1. CartPole 環境

- 台車を左右に押して、棒を立て続ける。棒が傾くか、台車が端に行くと終了する
- 状態は $(x,\dot x,\vartheta,\dot\vartheta)$ の4次元（$\vartheta$ は棒の角度）、行動は $\{0,1\}$（左に押す、右に押す）の2値
- 立っている間は毎ステップ報酬 $+1$ が得られる。500ステップ経っても倒れなければ、そこで打ち切る

次のセルが環境の実装である。
定数と運動方程式は OpenAI Gym の `CartPole-v1` と同じで、中身は読まなくてよい（「コードを表示」で中身が見られる）。
$n$ 個の CartPole を1つのテンソルにまとめて、同時に進める。

In [ ]:
# @title CartPole 環境
import math


class CartPole:
    """n 個の CartPole を同時に進める. 状態は (x, x', θ, θ'), 行動は {0: 左, 1: 右}.

    500 ステップ (max_steps) の打ち切りは環境の返り値ではなく, 呼ぶ側のループ回数で扱う.
    """

    def __init__(self, n, device="cpu", max_steps=500):
        self.n, self.device, self.max_steps = n, device, max_steps
        self.gravity, self.mass_cart, self.mass_pole = 9.8, 1.0, 0.1
        self.length, self.force_mag, self.tau = 0.5, 10.0, 0.02
        self.total_mass = self.mass_cart + self.mass_pole
        self.pole_mass_length = self.mass_pole * self.length
        self.x_threshold = 2.4
        self.theta_threshold = 12 * 2 * math.pi / 360

    def reset(self):
        self.state = (torch.rand(self.n, 4, device=self.device) - 0.5) * 0.1
        return self.state.clone()

    def step(self, action):
        x, x_dot, theta, theta_dot = self.state.unbind(1)
        force = torch.where(action == 1, self.force_mag, -self.force_mag)
        cos_t, sin_t = torch.cos(theta), torch.sin(theta)
        temp = (force + self.pole_mass_length * theta_dot**2 * sin_t) / self.total_mass
        theta_acc = (self.gravity * sin_t - cos_t * temp) / (
            self.length * (4.0 / 3.0 - self.mass_pole * cos_t**2 / self.total_mass)
        )
        x_acc = temp - self.pole_mass_length * theta_acc * cos_t / self.total_mass
        x = x + self.tau * x_dot
        x_dot = x_dot + self.tau * x_acc
        theta = theta + self.tau * theta_dot
        theta_dot = theta_dot + self.tau * theta_acc
        self.state = torch.stack([x, x_dot, theta, theta_dot], 1)
        terminated = (x.abs() > self.x_threshold) | (theta.abs() > self.theta_threshold)
        reward = torch.ones(self.n, device=self.device)  # 立っている限り毎ステップ +1
        return self.state.clone(), reward, terminated

### 環境の使い方

環境は `reset` と `step` の2つのメソッドを持つ。

- `s = env.reset()`：初期状態 `[n, 4]` を返す
- `s_next, r, terminated = env.step(a)`：行動 `a`（長さ $n$）を与えて1ステップ進める。次の状態 `[n, 4]`、報酬 `[n]`、終了したか `[n]` が返る

In [ ]:
env = CartPole(n=3)                   # 3 個の CartPole をまとめて扱う
s = env.reset()
print("s =", s)
a = torch.tensor([0, 1, 1])           # 環境ごとの行動
s_next, r, terminated = env.step(a)
print("s_next =", s_next)
print("r =", r, " terminated =", terminated)

### 1つのエピソードをアニメーションで見る

`rollout(act)` は1個の環境を、行動を選ぶ関数 `act` で終了（または500ステップ）まで動かし、状態と行動の列を返す。
`animate` はその結果を台車と棒のアニメーションにする。
矢印は、そのステップで台車を押した向きである。
破線の外に台車が出るか、棒が12度より傾くと終了する。

In [ ]:
# @title 可視化用の関数 rollout, animate
@torch.no_grad()
def rollout(act, max_steps=500):
    """1 個の環境を act(s) で動かし, 状態 [T+1, 4] と行動 [T] を numpy 配列で返す."""
    env = CartPole(n=1, max_steps=max_steps)
    s = env.reset()
    states, actions = [s[0]], []
    for _ in range(env.max_steps):
        a = act(s)
        s, r, terminated = env.step(a)
        states.append(s[0])
        actions.append(a[0])
        if terminated.item():
            break
    return torch.stack(states).numpy(), torch.stack(actions).numpy()


def animate(states, actions, title="", values=None, every=2, slow=1):
    """rollout の結果をアニメーションにする. every ステップごとに 1 コマ描き, slow 倍に遅くして再生する.

    values を渡すと各時刻の V(s_t) も表示する.
    """
    pole_len, cart_w, cart_h = 1.0, 0.5, 0.3  # 棒の長さは 2 × 0.5
    fig, ax = plt.subplots(figsize=(6, 2.3), dpi=80)
    ax.set_xlim(-2.9, 2.9)
    ax.set_ylim(-0.2, 1.4)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(title, fontsize=11)
    ax.plot([-2.9, 2.9], [0, 0], color="0.5", lw=1)
    for x in (-2.4, 2.4):  # 台車がこの外に出ると終了
        ax.plot([x, x], [-0.15, 0.7], color="0.6", ls="--", lw=1)
    cart = plt.Rectangle((0, 0), cart_w, cart_h, color="0.3")
    ax.add_patch(cart)
    (pole,) = ax.plot([], [], color="#E69F00", lw=6, solid_capstyle="round")
    arrow = ax.annotate("", xy=(0, 0), xytext=(0, 0), arrowprops=dict(arrowstyle="->", color="#0072B2", lw=2))
    text = ax.text(-2.85, 1.35, "", fontsize=11, va="top", family="monospace")

    def draw(t):
        x, _, th, _ = states[t]
        cart.set_xy((x - cart_w / 2, 0))
        pole.set_data([x, x + pole_len * np.sin(th)], [cart_h, cart_h + pole_len * np.cos(th)])
        if t < len(actions):
            d = 1 if actions[t] == 1 else -1
            arrow.set_position((x + d * cart_w / 2, cart_h / 2))  # 矢印の根元
            arrow.xy = (x + d * (cart_w / 2 + 0.4), cart_h / 2)   # 矢印の先
            arrow.set_visible(True)
        else:
            arrow.set_visible(False)
        label = f"t = {t:3d}"
        if values is not None and t < len(values):
            label += f"   V(s) = {values[t]:5.1f}"
        if t == len(actions) and len(actions) < 500:
            label += "   terminated"
        text.set_text(label)
        return cart, pole, arrow, text

    frames = list(range(0, len(states), every))
    if frames[-1] != len(states) - 1:
        frames.append(len(states) - 1)
    anim = animation.FuncAnimation(fig, draw, frames=frames, interval=20 * every * slow, repeat=False)  # slow=1 で実時間
    plt.close(fig)
    try:
        return HTML(anim.to_html5_video())  # 動画にする (ffmpeg が必要. Colab には入っている)
    except RuntimeError:  # ffmpeg が無い環境では, コマを並べた HTML にする
        return HTML(anim.to_jshtml(default_mode="once"))

まず、行動を毎回ランダムに選んで動かす。
20ステップほどで棒が倒れる。
短いので、実際の4分の1の速さで再生する。
再生ボタンで何度でも再生できる。

In [ ]:
torch.manual_seed(2)
random_act = lambda s: torch.randint(0, 2, (s.shape[0],))
states, actions = rollout(random_act)
print(f"{len(actions)} ステップで終了")
animate(states, actions, title="random policy", every=1, slow=4)

## 2. REINFORCE の実装

### 方策 NN

方策は状態 $s$ を受け取り、2つの行動のロジットを出力する NN である。
`Categorical(logits=...)` が softmax で確率に変換する。
前期のソフトマックス回帰と同じ構造である。

In [ ]:
def mlp(n_in, n_out, hidden=64):
    return nn.Sequential(
        nn.Linear(n_in, hidden), nn.Tanh(),
        nn.Linear(hidden, hidden), nn.Tanh(),
        nn.Linear(hidden, n_out))


torch.manual_seed(0)
policy = mlp(4, 2)  # 状態 s -> 行動の対数確率（の元になるロジット）
s = CartPole(n=3).reset()
dist = Categorical(logits=policy(s))
print("π(a|s) =", dist.probs)  # 学習前はほぼ 0.5 ずつ
print("サンプルした行動:", dist.sample())

### データ収集：M エピソードを最後まで

1回の更新に使う $M$ 本のエピソードを、まとめて並列に進める。
行動を選ぶとき、方策は $M$ 個の状態を1つのテンソルとして受け取る。

- 返り値はどれも `[T, M]` の形（状態は `[T, M, 4]`）。`S_next` と `TERM` は第4節の TD 学習で使う
- `ALIVE[t, i]` は、エピソード $i$ が時刻 $t$ にまだ続いているか。終了後の時刻は損失から除く
- 打ち切りは終了ではないので `ALIVE` は変えない。500ステップ目まで立っていたエピソードは、最後の時刻まで全てデータに入る

In [ ]:
@torch.no_grad()
def collect_episodes(env, policy):
    s = env.reset()
    alive = torch.ones(env.n, dtype=torch.bool)
    S, A, R, S_next, ALIVE, TERM = [], [], [], [], [], []
    for _ in range(env.max_steps):                  # 500 ステップで打ち切る
        a = Categorical(logits=policy(s)).sample()  # 方策から行動をサンプル
        s_next, r, terminated = env.step(a)
        S.append(s), A.append(a), R.append(r), S_next.append(s_next)
        ALIVE.append(alive), TERM.append(terminated)
        alive = alive & ~terminated
        s = s_next
        if not alive.any():
            break
    return [torch.stack(x) for x in (S, A, R, S_next, ALIVE, TERM)]

学習前の方策で $M=16$ 本集め、`ALIVE` を図にする。
横軸が時刻 $t$、縦軸がエピソード $i$ で、色の付いた部分がエピソードの続いている時刻である。
エピソードの長さはばらばらなので、テンソルの長さ $T$ は最も長いエピソードに合わせ、短いエピソードの後ろは `ALIVE` で除く。

In [ ]:
torch.manual_seed(0)
env = CartPole(n=16)
S, A, R, S_next, ALIVE, TERM = collect_episodes(env, policy)
print("S:", tuple(S.shape), " A:", tuple(A.shape), " ALIVE:", tuple(ALIVE.shape))
print("各エピソードの長さ:", ALIVE.sum(0).tolist())

fig, ax = plt.subplots(figsize=(6, 2.2))
ax.imshow(ALIVE.T.float(), aspect="auto", cmap="Blues", vmin=0, vmax=1.3, interpolation="nearest")
ax.set_xlabel("time step $t$")
ax.set_ylabel("episode $i$")
plt.show()

### 収益の計算：最後から順に

$$
G_{T_i}^{(i)}=0,\qquad
G_t^{(i)}=r_{t+1}^{(i)}+\gamma\,G_{t+1}^{(i)}
$$

- 後ろから足していけば、各時刻の収益（reward-to-go）が1回の走査で求まる
- `ALIVE[t]` を掛けると、終了したエピソードの収益は0のまま止まる

In [ ]:
def discounted_returns(R, ALIVE, gamma):
    G = torch.zeros_like(R)
    running = torch.zeros(R.shape[1])
    for t in reversed(range(R.shape[0])):
        running = (R[t] + gamma * running) * ALIVE[t]
        G[t] = running
    return G


G = discounted_returns(R, ALIVE, gamma=0.99)

fig, ax = plt.subplots(figsize=(4.5, 2.6))
for i in range(4):
    T_i = int(ALIVE[:, i].sum())
    ax.plot(range(T_i), G[:T_i, i], marker=".", ms=4, label=f"episode {i} ($T_i$ = {T_i})")
ax.set_xlabel("time step $t$")
ax.set_ylabel("$G_t$")
ax.legend(frameon=False)
plt.show()

どのエピソードも、最後の時刻で $G_t=1$ になり、前の時刻ほど大きい。
報酬は毎ステップ $+1$ なので、$\gamma=0.99$ で割り引いた収益は、500ステップ立ち続けても $\sum_{k=0}^{499}0.99^k\approx99.3$ にしかならない。
以下の学習曲線に示す「収益」は割引なしの報酬の和（立っていたステップ数、最大500）で、学習に使う $G_t$ とは別物である。

### 損失と更新：収益で重み付けした NLL

$$
\widehat{\mathcal L}_{\mathrm{RF}}(\theta)
=-\frac{1}{M}\sum_{i=1}^M\sum_{t=0}^{T_i-1}
\underbrace{\gamma^t\bigl(G_t^{(i)}-b(s_t^{(i)})\bigr)}_{W_t^{(i)}}
\log\pi_\theta(a_t^{(i)}|s_t^{(i)})
$$

まずはベースラインを使わず、$b(s_t)=0$ として動かす。
このとき重みは $W_t^{(i)}=\gamma^tG_t^{(i)}$ である。
`W.detach()` で重みを定数として扱うので、微分されるのは方策の対数尤度だけになる。

In [ ]:
torch.manual_seed(0)
policy = mlp(4, 2)
opt = torch.optim.Adam(policy.parameters(), lr=1e-3)
gamma = 0.99

S, A, R, S_next, ALIVE, TERM = collect_episodes(CartPole(n=16), policy)
G = discounted_returns(R, ALIVE, gamma)
discount_t = gamma ** torch.arange(len(R)).unsqueeze(1)  # γ^t, 形は [T, 1]

baseline = 0                                       # ベースラインなし (b = 0)
W = discount_t * (G - baseline)                    # 重み
logp = Categorical(logits=policy(S)).log_prob(A)   # log π_θ(a_t | s_t)
loss = -(W.detach() * logp * ALIVE).sum(0).mean()  # 時刻で和、エピソードで平均
opt.zero_grad(); loss.backward(); opt.step()
print("loss =", loss.item())

### 学習ループ

上の1回の更新を繰り返す。
各更新のあと、更新後の方策で新しいエピソードを集め直す（on-policy）。

後で勾配のばらつきを比べるため、10更新ごとに `grad_variance` も測る。
これは、各エピソード $i$ だけから計算した勾配 $g_i$ の、平均 $\bar g$ まわりのばらつき $\frac1M\sum_i\|g_i-\bar g\|^2$ である。
測るだけで、学習には使わない。

In [ ]:
# @title 勾配のばらつきを測る関数 grad_variance
def grad_variance(policy, S, A, W, ALIVE):
    """各エピソード i だけから計算した勾配 g_i の, 平均 ḡ まわりのばらつき (1/M) Σ_i ||g_i - ḡ||²."""
    logp = Categorical(logits=policy(S)).log_prob(A)
    losses = -(W.detach() * logp * ALIVE).sum(0)  # エピソードごとの損失 [M]
    params = list(policy.parameters())
    grads = []
    for loss_i in losses:
        g = torch.autograd.grad(loss_i, params, retain_graph=True)
        grads.append(torch.cat([x.reshape(-1) for x in g]))
    g = torch.stack(grads)  # [M, パラメータ数]
    return (g - g.mean(0)).pow(2).sum(1).mean().item()

In [ ]:
def train_reinforce(seed, M=16, updates=UPDATES, gamma=0.99, lr=1e-3):
    torch.manual_seed(seed)
    env = CartPole(n=M)
    policy = mlp(4, 2)
    opt = torch.optim.Adam(policy.parameters(), lr=lr)
    log = {"env_steps": [], "return": [], "grad_var": []}
    env_steps = 0
    for update in range(1, updates + 1):
        S, A, R, S_next, ALIVE, TERM = collect_episodes(env, policy)
        env_steps += int(ALIVE.sum())                      # 環境とやり取りしたステップ数
        G = discounted_returns(R, ALIVE, gamma)
        discount_t = gamma ** torch.arange(len(R)).unsqueeze(1)

        baseline = 0                                       # ベースラインなし (b = 0)
        W = discount_t * (G - baseline)
        if update % 10 == 0:
            log["grad_var"].append((env_steps, grad_variance(policy, S, A, W, ALIVE)))
        logp = Categorical(logits=policy(S)).log_prob(A)
        loss = -(W.detach() * logp * ALIVE).sum(0).mean()
        opt.zero_grad(); loss.backward(); opt.step()

        log["env_steps"].append(env_steps)
        log["return"].append((R * ALIVE).sum(0).mean().item())  # 割引なしの収益の平均
        if update % 100 == 0:
            print(f"  update {update}  env steps {env_steps:7d}  return {log['return'][-1]:5.1f}")
    return {"policy": policy, "log": log}


def run_seeds(train, **kwargs):
    """シード 0, 1, ..., N_SEEDS-1 で train を実行し, 結果をリストで返す."""
    runs = []
    for seed in range(N_SEEDS):
        t0 = time.time()
        print(f"seed {seed}")
        runs.append(train(seed, **kwargs))
        print(f"  ({time.time() - t0:.0f} 秒)")
    return runs

### 学習の設定と実行

| 項目 | 値 |
|:--|:--|
| 1回の更新に使うエピソード数 $M$ | 16 |
| 更新回数 | 300 |
| 割引率 $\gamma$ | 0.99 |
| 方策の最適化 | Adam、学習率 $10^{-3}$ |
| 乱数シード | `N_SEEDS` 通り |

同じ設定でも乱数シードによって結果は変わるため、複数のシードで学習して比べる。

In [ ]:
results = {}  # 手法ごとに, シードごとの学習結果を入れる
results["none"] = run_seeds(train_reinforce)

### 結果：REINFORCE の学習曲線

横軸は環境との相互作用の回数（環境ステップ数）、縦軸は1回の更新に使った16エピソードの平均収益である。
細い線は各シード、太い線はシードの平均である。

In [ ]:
# @title 学習曲線を描く関数 plot_curves
YLABEL = {"return": "return", "grad_var": "gradient variance", "critic_rmse": "critic RMSE"}


def series(log, field):
    """log から (環境ステップ数, 値) の列を取り出す."""
    if field == "grad_var":
        x, y = zip(*log["grad_var"])
    else:
        x, y = log["env_steps"], log[field]
    return np.array(x, dtype=float), np.array(y, dtype=float)


def plot_curves(keys, field="return", ax=None, band=True):
    """band=True なら平均 (線) と最小〜最大の範囲 (帯), False なら各シード (細線) と平均 (太線)."""
    if ax is None:
        _, ax = plt.subplots(figsize=(4.5, 2.8))
    for key in keys:
        xs, ys = zip(*(series(run["log"], field) for run in results[key]))
        grid = np.linspace(max(x[0] for x in xs), min(x[-1] for x in xs), 200)
        Y = np.array([np.interp(grid, x, y) for x, y in zip(xs, ys)])  # シード間で横軸を揃える
        if band:
            ax.fill_between(grid, Y.min(0), Y.max(0), color=COLOR[key], alpha=0.2, lw=0)
        else:
            for x, y in zip(xs, ys):
                ax.plot(x, y, color=COLOR[key], alpha=0.4, lw=0.7)
        ax.plot(grid, Y.mean(0), color=COLOR[key], ls=LS[key], lw=2, label=LABEL[key])
    ax.set_xlabel("environment steps")
    ax.set_ylabel(YLABEL[field])
    if field == "return":
        ax.axhline(500, color="0.6", lw=0.8, ls=":")
        ax.set_ylim(0, 540)
    if field == "grad_var":
        ax.set_yscale("log")
    ax.legend(frameon=False, fontsize=9)
    return ax

In [ ]:
plot_curves(["none"], band=False)
plt.show()

- 学習は進み、収益が上限の500に届く
- しかし収益の上下が大きく、一度500に達しても大きく落ち込むことがある。シード間の差も大きい

### 学習した方策を見る

最後の更新での収益が最も高かったシードの方策で、1エピソード動かす。
方策は確率的なので、実行するたびに結果が変わる。

In [ ]:
best = max(results["none"], key=lambda run: run["log"]["return"][-1])
policy_none = best["policy"]
torch.manual_seed(1)
states, actions = rollout(lambda s: Categorical(logits=policy_none(s)).sample())
print(f"{len(actions)} ステップ")
animate(states, actions, title="REINFORCE (no baseline)")

### 方策は何を学んだか

台車が中央に止まっている状態（$x=\dot x=0$）で、棒の角度 $\vartheta$ と角速度 $\dot\vartheta$ を変えたときの、右に押す確率 $\pi(1\mid s)$ を色で示す。
左図は学習前、右図は学習後の方策である。

In [ ]:
def plot_policy_map(policy, ax, title=""):
    th, thd = np.meshgrid(np.linspace(-0.21, 0.21, 101), np.linspace(-1.5, 1.5, 101))
    s = torch.zeros(th.size, 4)
    s[:, 2], s[:, 3] = torch.tensor(th.ravel()), torch.tensor(thd.ravel())
    with torch.no_grad():
        p_right = Categorical(logits=policy(s)).probs[:, 1].reshape(th.shape)
    im = ax.pcolormesh(th, thd, p_right, cmap="RdBu_r", vmin=0, vmax=1, shading="auto")
    ax.set_xlabel(r"pole angle $\vartheta$ [rad]")
    ax.set_ylabel(r"angular velocity $\dot\vartheta$")
    ax.set_title(title, fontsize=10)
    return im


torch.manual_seed(0)
fig, axes = plt.subplots(1, 2, figsize=(8, 2.8), constrained_layout=True)
plot_policy_map(mlp(4, 2), axes[0], "before training")
im = plot_policy_map(policy_none, axes[1], "after training")
fig.colorbar(im, ax=axes, label="P(push right)")
plt.show()

学習前の方策は、どの状態でもほぼ半々の確率で押す向きを選ぶ。
学習後の方策は、棒が右に倒れつつある（$\dot\vartheta>0$）ときや右に傾いている（$\vartheta>0$）ときに、右へ押す確率が高い。
台車を棒の倒れる側へ動かして支える動きを、報酬だけから学習している。

## 3. 価値予測とベースライン

### なぜ学習が不安定になるのか

- 勾配の推定量は、たった $M$ 本の軌跡から作ったモンテカルロ推定である
- 収益 $G_t$ には「その行動の良さ」だけでなく「状態の有利さ」と「その後の偶然」が混ざる
- 行動に依存しないベースライン $b(s_t)$ を引けば、勾配の期待値を変えずにばらつきを減らせる

そこで、状態価値 $V^\pi(s)=\mathbb E_\pi[G_t\mid s_t=s]$ を NN $V_\phi(s)$（**critic**）で表し、データから学習してベースラインに使う。

### MC 収益への回帰

集めた軌跡から、状態と実際に得た収益の組 $\{(s_t^{(i)},G_t^{(i)})\}$ を作り、二乗誤差を最小化する。

$$
\mathcal L(\phi)
=\frac{1}{N}\sum_{(s,G)}\bigl(V_\phi(s)-G\bigr)^2,
\qquad N=\sum_i T_i
$$

- ターゲット $G_t$ はモンテカルロ（MC）推定である。$\mathbb E[G_t\mid s_t]=V^\pi(s_t)$ なので、ノイズはあるが偏りはない
- 状態が連続なので、同じ状態を2度訪れることはまずない。状態ごとに収益を平均するのではなく、1つの関数を全データに合わせる

### ベースラインなしの実装との差分

`train_reinforce` との違いは `# ★` を付けた行である。

- 重みを $W_t^{(i)}=\gamma^t\bigl(G_t^{(i)}-V_\phi(s_t^{(i)})\bigr)$ に変える
- `baseline` は、このバッチで critic を学習する前に計算する。学習後の $V_\phi$ はバッチ内の行動に依存し、不偏性が崩れる
- `baseline` は critic の出力だが、`W.detach()` により actor の損失の勾配は critic に流れない
- critic の学習ターゲットは、引数 `critic_target` で MC（`"mc"`）と TD（`"td"`）を切り替える。TD は第4節で使う

In [ ]:
def train_with_critic(seed, critic_target="mc", M=16, updates=UPDATES, gamma=0.99, lr=1e-3,
                      lr_critic=3e-3, critic_steps=20):
    torch.manual_seed(seed)
    env = CartPole(n=M)
    policy = mlp(4, 2)
    critic = mlp(4, 1)                                           # ★ V_φ(s)
    opt = torch.optim.Adam(policy.parameters(), lr=lr)
    opt_v = torch.optim.Adam(critic.parameters(), lr=lr_critic)  # ★
    log = {"env_steps": [], "return": [], "grad_var": [], "critic_rmse": []}
    env_steps = 0
    for update in range(1, updates + 1):
        S, A, R, S_next, ALIVE, TERM = collect_episodes(env, policy)
        env_steps += int(ALIVE.sum())
        G = discounted_returns(R, ALIVE, gamma)
        discount_t = gamma ** torch.arange(len(R)).unsqueeze(1)

        baseline = critic(S).squeeze(-1)                         # ★ V_φ(s_t)
        W = discount_t * (G - baseline)
        if update % 10 == 0:
            log["grad_var"].append((env_steps, grad_variance(policy, S, A, W, ALIVE)))
        logp = Categorical(logits=policy(S)).log_prob(A)
        loss = -(W.detach() * logp * ALIVE).sum(0).mean()
        opt.zero_grad(); loss.backward(); opt.step()             # actor の更新

        with torch.no_grad():                                    # ★ critic の学習ターゲット (定数)
            if critic_target == "mc":
                target = G                                       # ★ MC ターゲット
            else:
                v_next = critic(S_next).squeeze(-1) * (~TERM)    # ★ 終端なら 0
                target = R + gamma * v_next                      # ★ TD ターゲット
        for _ in range(critic_steps):                            # ★ critic の更新
            v = critic(S).squeeze(-1)                            # ★
            loss_v = ((v - target) ** 2 * ALIVE).sum() / ALIVE.sum()  # ★
            opt_v.zero_grad(); loss_v.backward(); opt_v.step()   # ★

        with torch.no_grad():  # 実際の収益 G_t に対する critic の誤差 (RMSE) を記録する
            v = critic(S).squeeze(-1)
            log["critic_rmse"].append(((v - G) ** 2 * ALIVE).sum().div(ALIVE.sum()).sqrt().item())
        log["env_steps"].append(env_steps)
        log["return"].append((R * ALIVE).sum(0).mean().item())
        if update % 100 == 0:
            print(f"  update {update}  env steps {env_steps:7d}  return {log['return'][-1]:5.1f}"
                  f"  critic RMSE {log['critic_rmse'][-1]:5.1f}")
    return {"policy": policy, "critic": critic, "log": log}

In [ ]:
results["mc"] = run_seeds(train_with_critic, critic_target="mc")

### 結果：ベースラインの効果

線はシードの平均、帯は最小から最大の範囲である。
左図は学習曲線である。
右図は勾配のばらつき $\frac1M\sum_i\|g_i-\bar g\|^2$ で、縦軸は対数である。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 2.8), constrained_layout=True)
plot_curves(["none", "mc"], "return", ax=axes[0])
plot_curves(["none", "mc"], "grad_var", ax=axes[1])
plt.show()

- 左：価値ベースラインを入れると、500に到達したあとの落ち込みが小さく、シード間の差も小さい
- 右：ベースラインを入れると、エピソードごとの勾配のばらつきが小さくなる。ベースラインなしでは学習が進むほどばらつきが大きくなり、終盤では2桁ほどの差がつく
- 1回の更新に使うエピソード数は同じである。違いは重みの作り方と、critic の学習（1回の更新につき20ステップの回帰）が加わる点である
- ベースラインは勾配の期待値を変えないので、減ったのはばらつきだけである

### 価値の予測はどのくらい当たるか

左図は、学習中の critic の予測 $V_\phi(s_t)$ と実際の収益 $G_t$ との誤差（RMSE）の推移である。
右図は、学習後の方策で新しく集めた16エピソードについて、実際の収益と critic の予測を比べた散布図である（シード0）。

In [ ]:
run = results["mc"][0]
torch.manual_seed(100)
S, A, R, S_next, ALIVE, TERM = collect_episodes(CartPole(n=16), run["policy"])
G = discounted_returns(R, ALIVE, 0.99)
with torch.no_grad():
    v = run["critic"](S).squeeze(-1)
G_alive, v_alive = G[ALIVE], v[ALIVE]
idx = torch.randperm(len(G_alive))[:1000]  # 点が多すぎるので 1000 点だけ描く

fig, axes = plt.subplots(1, 2, figsize=(9, 2.8), constrained_layout=True)
plot_curves(["mc"], "critic_rmse", ax=axes[0])
axes[0].set_ylim(0, None)
axes[1].scatter(G_alive[idx], v_alive[idx], s=5, alpha=0.4, color=COLOR["mc"], lw=0)
axes[1].plot([0, 105], [0, 105], color="0.5", ls="--", lw=1)
axes[1].set_xlabel("actual return $G_t$")
axes[1].set_ylabel(r"prediction $V_\phi(s_t)$")
axes[1].set_aspect("equal")
plt.show()

- 左：収益には方策のランダム性によるばらつきが残るので、誤差は0にはならない。方策が変わると $V^\pi$ も変わるので、critic は動く目標を追いかけ続ける
- 右：予測が実測を大きく上回る点（左上の点）は、実際の収益 $G_t$ が小さい、500ステップの打ち切り直前の状態である。残り時間は状態に含まれていないので、critic にはほかの状態と区別しにくい

### 1つのエピソードに沿って価値の予測を見る

学習後の方策で1エピソード動かし、各時刻の予測 $V_\phi(s_t)$ を表示する。
下の図は、その予測と実際の収益 $G_t$ を時刻に沿って並べたものである。

In [ ]:
policy_mc, critic_mc = run["policy"], run["critic"]
torch.manual_seed(1)
states, actions = rollout(lambda s: Categorical(logits=policy_mc(s)).sample())
T = len(actions)
with torch.no_grad():
    v_ep = critic_mc(torch.tensor(states[:-1])).squeeze(-1).numpy()  # V_φ(s_0), ..., V_φ(s_{T-1})
G_ep = (1 - 0.99 ** (T - np.arange(T))) / (1 - 0.99)  # 報酬が毎ステップ 1 のときの G_t

fig, ax = plt.subplots(figsize=(5, 2.6))
ax.plot(G_ep, color="0.3", label="actual return $G_t$")
ax.plot(v_ep, color=COLOR["mc"], ls="--", label=r"prediction $V_\phi(s_t)$")
ax.set_xlabel("time step $t$")
ax.set_ylim(0, 110)
ax.legend(frameon=False, fontsize=9)
plt.show()
animate(states, actions, title="value baseline (MC)", values=v_ep)

500ステップまで立ち続けた場合、次のことが見える。

- 実際の収益 $G_t$ は、終わりに近づくと急に0へ下がる。500ステップで打ち切るので、その先の報酬が入らないためである
- critic の予測は、終わり際の急な下がり方に追いつかない。状態に残り時間が含まれていないためである
- 打ち切りの先を価値の予測で補う方法は、次回の補足で扱う

## 4. TD ターゲットで critic を学習する

TD(0) のターゲット $y_t=r_{t+1}+\gamma V_\phi(s_{t+1})$ は、エピソードの終了を待たずに、直後の報酬と現在の予測から作る。

- 終端状態では $V_\phi(s_{t+1})=0$ とする。0にするのは棒が倒れた（`TERM`）ときだけである
- 500ステップの打ち切りは終端ではないので、$V_\phi(s_{t+1})$ をそのまま使う。終了と打ち切りを区別する理由はここにある
- ターゲットは `torch.no_grad()` の中で計算し、定数として扱う

`train_with_critic` に `critic_target="td"` を渡すと、このターゲットで critic を学習する。
critic は REINFORCE のベースラインとしてだけ使い、actor の重みは $\gamma^t(G_t-V_\phi(s_t))$ のまま変えない。

In [ ]:
results["td"] = run_seeds(train_with_critic, critic_target="td")

### 結果：ターゲットの違いが critic と学習に与える影響

左図は学習曲線、右図は実際の収益 $G_t$ に対する critic の誤差である。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 2.8), constrained_layout=True)
plot_curves(["mc", "td"], "return", ax=axes[0])
plot_curves(["mc", "td"], "critic_rmse", ax=axes[1])
axes[1].set_ylim(0, None)
plt.show()

- 左：ベースラインの誤差は勾配の期待値を変えないので、どちらの critic でも方策の学習は進む
- 右：TD の critic は、収益が急に伸びる学習の序盤に、実際の収益から大きくずれる。方策の変化で $V^\pi$ が動くとき、自分の予測の誤差がターゲットに入り、ブートストラップで伝わるためである
- 右図の指標は「MC 収益との差」なので、MC ターゲットに回帰した critic に有利である。TD が小さくするのは、1ステップ先の予測との差である

## 5. 発展

設定を変えて、結果がどう変わるかを確かめてみよう。
`N_SEEDS = 1` にすると早く終わる。

- 1回の更新に使うエピソード数を変える（例：`run_seeds(train_reinforce, M=4)`）。学習曲線と勾配のばらつきはどう変わるか
- `critic_steps` を1や100に変える（例：`run_seeds(train_with_critic, critic_steps=1)`）。ベースラインの効果はどう変わるか
- 重みの $\gamma^t$ を外す（`discount_t` を1にする）と、学習曲線はどう変わるか
- TD 誤差 $\delta_t=r_{t+1}+\gamma V_\phi(s_{t+1})-V_\phi(s_t)$ をそのまま actor の重みにすると Actor–Critic になる。`train_with_critic` を書き換えて試す（講義では次回、GAE の $\lambda=0$ の場合として扱う）